In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from decimal import Decimal
from enum import Enum
from typing import Optional


# ─────────────────────────────────────────────
# VALUE OBJECTS
# ─────────────────────────────────────────────

@dataclass(frozen=True)
class Dinheiro:
    valor: Decimal

    def __post_init__(self):
        if self.valor < 0:
            raise ValueError(f"Valor monetário não pode ser negativo: {self.valor}")

    def __add__(self, outro: Dinheiro) -> Dinheiro:
        return Dinheiro(self.valor + outro.valor)

    def __sub__(self, outro: Dinheiro) -> Dinheiro:
        resultado = self.valor - outro.valor
        return Dinheiro(max(resultado, Decimal("0")))

    def __mul__(self, fator: Decimal) -> Dinheiro:
        return Dinheiro((self.valor * fator).quantize(Decimal("0.01")))

    def __gt__(self, outro: Dinheiro) -> bool:
        return self.valor > outro.valor

    def __repr__(self):
        return f"R$ {self.valor:.2f}"


@dataclass(frozen=True)
class Percentual:
    valor: Decimal  # 0 a 100

    def __post_init__(self):
        if not (Decimal("0") <= self.valor <= Decimal("100")):
            raise ValueError(f"Percentual inválido: {self.valor}")

    def aplicar_sobre(self, dinheiro: Dinheiro) -> Dinheiro:
        return dinheiro * (self.valor / Decimal("100"))


@dataclass(frozen=True)
class CodigoCupom:
    valor: str

    def __post_init__(self):
        if not self.valor or len(self.valor) < 4:
            raise ValueError("Código de cupom deve ter ao menos 4 caracteres")
        object.__setattr__(self, "valor", self.valor.upper())


@dataclass(frozen=True)
class Quantidade:
    valor: int

    def __post_init__(self):
        if self.valor < 0:
            raise ValueError("Quantidade não pode ser negativa")

    def subtrair(self, outra: Quantidade) -> Quantidade:
        if outra.valor > self.valor:
            raise ValueError("Estoque insuficiente")
        return Quantidade(self.valor - outra.valor)

    def esta_disponivel(self) -> bool:
        return self.valor > 0


# ─────────────────────────────────────────────
# ENTIDADES DE DOMÍNIO
# ─────────────────────────────────────────────

@dataclass(frozen=True)
class Produto:
    nome: str
    preco: Dinheiro

    def __post_init__(self):
        if not self.nome.strip():
            raise ValueError("Produto precisa ter um nome")


class Estoque:
    def __init__(self):
        self._inventario: dict[Produto, Quantidade] = {}

    def adicionar(self, produto: Produto, quantidade: Quantidade) -> None:
        atual = self._inventario.get(produto, Quantidade(0))
        self._inventario[produto] = Quantidade(atual.valor + quantidade.valor)

    def reservar(self, produto: Produto, quantidade: Quantidade) -> None:
        disponivel = self._inventario.get(produto, Quantidade(0))
        self._inventario[produto] = disponivel.subtrair(quantidade)

    def disponivel_para(self, produto: Produto, quantidade: Quantidade) -> bool:
        atual = self._inventario.get(produto, Quantidade(0))
        return atual.valor >= quantidade.valor

    def quantidade_de(self, produto: Produto) -> Quantidade:
        return self._inventario.get(produto, Quantidade(0))


# ─────────────────────────────────────────────
# CUPONS
# ─────────────────────────────────────────────

class TipoCupom(Enum):
    PERCENTUAL = "percentual"
    VALOR_FIXO = "valor_fixo"


@dataclass(frozen=True)
class Cupom:
    codigo: CodigoCupom
    tipo: TipoCupom
    desconto: Decimal  # percentual (0-100) ou valor fixo

    def calcular_desconto(self, subtotal: Dinheiro) -> Dinheiro:
        if self.tipo == TipoCupom.PERCENTUAL:
            return Percentual(self.desconto).aplicar_sobre(subtotal)
        return Dinheiro(min(self.desconto, subtotal.valor))


class RepositorioDeCupons:
    def __init__(self):
        self._cupons: dict[str, Cupom] = {}

    def registrar(self, cupom: Cupom) -> None:
        self._cupons[cupom.codigo.valor] = cupom

    def buscar(self, codigo: CodigoCupom) -> Optional[Cupom]:
        return self._cupons.get(codigo.valor)


# ─────────────────────────────────────────────
# CARRINHO
# ─────────────────────────────────────────────

@dataclass
class ItemDoCarrinho:
    produto: Produto
    quantidade: Quantidade

    def subtotal(self) -> Dinheiro:
        return self.produto.preco * Decimal(self.quantidade.valor)


class ItensDoCarrinho:
    def __init__(self):
        self._itens: list[ItemDoCarrinho] = []

    def adicionar(self, item: ItemDoCarrinho) -> None:
        existente = self._buscar_produto(item.produto)
        if existente:
            existente.quantidade = Quantidade(
                existente.quantidade.valor + item.quantidade.valor
            )
            return
        self._itens.append(item)

    def remover(self, produto: Produto) -> None:
        self._itens = [i for i in self._itens if i.produto != produto]

    def subtotal(self) -> Dinheiro:
        return sum(
            (i.subtotal() for i in self._itens),
            Dinheiro(Decimal("0"))
        )

    def esta_vazio(self) -> bool:
        return len(self._itens) == 0

    def _buscar_produto(self, produto: Produto) -> Optional[ItemDoCarrinho]:
        for item in self._itens:
            if item.produto == produto:
                return item
        return None

    def __iter__(self):
        return iter(self._itens)


class StatusCarrinho(Enum):
    ABERTO = "aberto"
    FINALIZADO = "finalizado"
    CANCELADO = "cancelado"


class Carrinho:
    def __init__(self, estoque: Estoque, repositorio_cupons: RepositorioDeCupons):
        self._itens = ItensDoCarrinho()
        self._estoque = estoque
        self._repositorio_cupons = repositorio_cupons
        self._cupom: Optional[Cupom] = None
        self._status = StatusCarrinho.ABERTO

    def adicionar_produto(self, produto: Produto, quantidade: Quantidade) -> None:
        self._garantir_aberto()
        self._garantir_estoque(produto, quantidade)
        self._itens.adicionar(ItemDoCarrinho(produto, quantidade))

    def remover_produto(self, produto: Produto) -> None:
        self._garantir_aberto()
        self._itens.remover(produto)

    def aplicar_cupom(self, codigo: CodigoCupom) -> None:
        self._garantir_aberto()
        cupom = self._repositorio_cupons.buscar(codigo)
        if not cupom:
            raise ValueError(f"Cupom '{codigo.valor}' não encontrado")
        self._cupom = cupom

    def finalizar(self) -> Resumo:
        self._garantir_aberto()
        self._garantir_nao_vazio()
        self._reservar_estoque()
        self._status = StatusCarrinho.FINALIZADO
        return self._gerar_resumo()

    def cancelar(self) -> None:
        self._garantir_aberto()
        self._status = StatusCarrinho.CANCELADO

    def esta_finalizado(self) -> bool:
        return self._status == StatusCarrinho.FINALIZADO

    def total(self) -> Dinheiro:
        subtotal = self._itens.subtotal()
        if not self._cupom:
            return subtotal
        return subtotal - self._cupom.calcular_desconto(subtotal)

    # ── métodos privados ──

    def _garantir_aberto(self) -> None:
        if self._status != StatusCarrinho.ABERTO:
            raise ValueError(f"Carrinho está {self._status.value}, não pode ser modificado")

    def _garantir_nao_vazio(self) -> None:
        if self._itens.esta_vazio():
            raise ValueError("Não é possível finalizar um carrinho vazio")

    def _garantir_estoque(self, produto: Produto, quantidade: Quantidade) -> None:
        if not self._estoque.disponivel_para(produto, quantidade):
            disponivel = self._estoque.quantidade_de(produto)
            raise ValueError(
                f"Estoque insuficiente para '{produto.nome}': "
                f"pedido {quantidade.valor}, disponível {disponivel.valor}"
            )

    def _reservar_estoque(self) -> None:
        for item in self._itens:
            self._estoque.reservar(item.produto, item.quantidade)

    def _gerar_resumo(self) -> Resumo:
        subtotal = self._itens.subtotal()
        desconto = self._cupom.calcular_desconto(subtotal) if self._cupom else Dinheiro(Decimal("0"))
        return Resumo(
            itens=list(self._itens),
            subtotal=subtotal,
            desconto=desconto,
            total=subtotal - desconto,
            cupom_aplicado=self._cupom,
        )


# ─────────────────────────────────────────────
# RESUMO DO PEDIDO (output imutável)
# ─────────────────────────────────────────────

@dataclass(frozen=True)
class Resumo:
    itens: list
    subtotal: Dinheiro
    desconto: Dinheiro
    total: Dinheiro
    cupom_aplicado: Optional[Cupom]

    def imprimir(self) -> None:
        print("\n========== RESUMO DO PEDIDO ==========")
        for item in self.itens:
            print(f"  {item.produto.nome:20s} x{item.quantidade.valor}  {item.subtotal()}")
        print("--------------------------------------")
        print(f"  Subtotal:   {self.subtotal}")
        if self.cupom_aplicado:
            print(f"  Cupom:      -{self.desconto}  ({self.cupom_aplicado.codigo.valor})")
        print(f"  TOTAL:      {self.total}")
        print("======================================\n")


# ─────────────────────────────────────────────
# USO
# ─────────────────────────────────────────────

if __name__ == "__main__":
    # Infraestrutura
    estoque = Estoque()
    cupons = RepositorioDeCupons()

    # Produtos
    cafe = Produto("Café Especial 250g", Dinheiro(Decimal("32.90")))
    chá  = Produto("Chá Verde 100g",     Dinheiro(Decimal("18.50")))
    bolo = Produto("Bolo de Cenoura",    Dinheiro(Decimal("45.00")))

    # Populando estoque
    estoque.adicionar(cafe, Quantidade(10))
    estoque.adicionar(chá,  Quantidade(5))
    estoque.adicionar(bolo, Quantidade(2))

    # Registrando cupons
    cupons.registrar(Cupom(CodigoCupom("DESCONTO10"), TipoCupom.PERCENTUAL,  Decimal("10")))
    cupons.registrar(Cupom(CodigoCupom("FRETE20"),    TipoCupom.VALOR_FIXO,  Decimal("20")))

    # Montando carrinho
    carrinho = Carrinho(estoque, cupons)
    carrinho.adicionar_produto(cafe, Quantidade(2))
    carrinho.adicionar_produto(chá,  Quantidade(1))
    carrinho.adicionar_produto(bolo, Quantidade(1))
    carrinho.aplicar_cupom(CodigoCupom("desconto10"))  # aceita minúsculo

    # Finalizando
    resumo = carrinho.finalizar()
    resumo.imprimir()

    print(f"Estoque café restante: {estoque.quantidade_de(cafe).valor}")
    print(f"Carrinho finalizado:   {carrinho.esta_finalizado()}")
